# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [1]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

C:\ProgramData\miniconda3\envs\torch-class\Lib\site-packages\torch\_subclasses\functional_tensor.py:276: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [2]:
import torch

# 数据
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

# 创建三个待学习参数，初始化为 0
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)

# 使用 SGD 优化器，传入三个参数
optimizer = torch.optim.SGD([w1, w2, w3], lr=0.05)

epochs = 2000
loss_history = []

for epoch in range(1, epochs + 1):
    # 前向计算：预测值 = w1*x + w2*x² + w3
    y_pred = w1 * x + w2 * x ** 2 + w3
    
    # 损失：所有样本的平均绝对误差
    loss = torch.mean(torch.abs(y - y_pred))
    
    # 反向传播前清零梯度
    optimizer.zero_grad()
    loss.backward()
    
    # 学习率逐渐衰减，让后期更稳定
    lr = 0.05 / (1 + epoch * 0.001)
    optimizer.param_groups[0]["lr"] = lr
    
    # 更新参数
    optimizer.step()
    
    # 记录损失
    loss_history.append(loss.item())

# ===== 输出最终结果 =====
print("=" * 50)
print("训练完成！最终结果：")
print("=" * 50)
print(f"学到的 w1 = {w1.item():.6f}")
print(f"学到的 w2 = {w2.item():.6f}")
print(f"学到的 w3 = {w3.item():.6f}")
print()

# 最终预测值和损失
with torch.no_grad():
    y_pred_final = w1 * x + w2 * x ** 2 + w3
    final_loss = torch.mean(torch.abs(y - y_pred_final)).item()

print(f"最终平均绝对误差 = {final_loss:.6f}")
print()
print("每个样本的预测对比：")
print(f"{'x':>6}  {'真实y':>8}  {'预测y':>8}  {'绝对误差':>10}")
for i in range(len(x)):
    print(f"{x[i]:6.1f}  {y[i]:8.3f}  {y_pred_final[i]:8.4f}  {abs(y[i].item() - y_pred_final[i].item()):10.6f}")

print()
print("损失变化：")
print(f"  初始损失（第1轮）：{loss_history[0]:.6f}")
print(f"  最终损失（第2000轮）：{loss_history[-1]:.6f}")
print(f"  损失下降比例：{(1 - loss_history[-1]/loss_history[0])*100:.2f}%")


训练完成！最终结果：
学到的 w1 = 1.494913
学到的 w2 = -0.512504
学到的 w3 = 1.985186

最终平均绝对误差 = 0.035655

每个样本的预测对比：
     x       真实y       预测y        绝对误差
  -2.0    -3.000   -3.0547    0.054658
  -1.5    -1.375   -1.4103    0.035318
  -1.0     0.000   -0.0222    0.022231
  -0.5     1.125    1.1096    0.015397
   0.0     2.000    1.9852    0.014814
   0.5     2.625    2.6045    0.020484
   1.0     3.000    2.9676    0.032406
   1.5     3.125    3.0744    0.050580
   2.0     3.000    2.9250    0.075006

损失变化：
  初始损失（第1轮）：2.138889
  最终损失（第2000轮）：0.027308
  损失下降比例：98.72%


### 我的观察与说明

**1. 对作业 01 的训练代码做了哪些修改？**

从作业 01 到作业 02，主要做了以下几处修改：

1. **参数数量**：从 1 个参数 `w` 变成了 3 个参数 `w1, w2, w3`，优化器里传入的参数列表也从 `[w]` 变成了 `[w1, w2, w3]`。

2. **预测公式**：从简单的 `ŷ = x * w²` 变成了 `ŷ = w1*x + w2*x² + w3`，多了一次项、二次项和常数项的组合。

3. **数据规模**：从单个样本（x=1, y=4）变成了 9 个样本组成的张量，x 和 y 都是长度为 9 的向量。

4. **损失计算**：从单个样本的绝对值误差 `|y - y_pred|` 变成了所有样本损失的平均值，用 `torch.mean(torch.abs(y - y_pred))` 来计算。

5. **学习率策略**：因为参数变多了，我用了稍微温和一点的初始学习率（0.05），并加上了学习率衰减，让训练后期更稳定。

---

**2. 如何判断这次训练有效？**

我从三个方面判断训练是有效的：

1. **损失明显下降**：初始损失是 2.14，训练到 2000 轮后降到了 0.036，下降了超过 98%，说明模型在不断学习拟合数据。

2. **参数接近理论值**：手算理论解应该是 w1=1.5, w2=-0.5, w3=2.0。训练出来的结果是 w1≈1.495, w2≈-0.513, w3≈1.985，三个参数都和理论值非常接近，误差很小。

3. **预测值误差很小**：逐个对比 9 个样本的预测值和真实值，每个样本的绝对误差都在 0.08 以内，大部分都在 0.05 以下，说明拟合效果很好。

（因为用的是平均绝对误差损失，在最优解附近会有微小的震荡，所以参数不会精确等于理论值，这是正常现象。）


## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。